# Tugas 4: Implementasi Blockchain Sederhana dengan Python
**Mata Kuliah:** IF219 Teknologi Blockchain  
**Program Studi:** S1 Teknologi Informasi  
**Institut Teknologi Tangerang Selatan (ITTS)**  

---
**Identitas Mahasiswa:**
- **Nama:** Rafael Bramantyo Buana Putra
- **NIM:** 1002240003
- **Pertemuan:** 6 (Tugas Individu - Bobot 5%)
- **Sub-CPMK:** SUB-IF219-2-3 Menerapkan mekanisme pembentukan blok, transaksi, dan validasi rantai pada program blockchain sederhana.

## 1. Definisi Struktur Class `Block` dan `Blockchain`
Menyiapkan class `Block` dengan hash SHA-256 deterministik dan class `Blockchain` dengan inisialisasi Genesis Block otomatis.

In [1]:
import hashlib
import json
import time

class Block:
    def __init__(self, index, data, previous_hash, nonce=0):
        self.index = index
        self.timestamp = time.time()
        self.data = data
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        payload = {
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }
        raw = json.dumps(payload, sort_keys=True).encode()
        return hashlib.sha256(raw).hexdigest()

    def __repr__(self):
        return f"Block(Index={self.index}, Hash={self.hash[:16]}..., PrevHash={self.previous_hash[:16]}...)"


class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]

    def create_genesis_block(self):
        return Block(
            index=0,
            data={"keterangan": "Genesis Block ITTS Blockchain"},
            previous_hash="0" * 64
        )

    def latest_block(self):
        return self.chain[-1]

    def add_block(self, data):
        prev_block = self.latest_block()
        new_block = Block(
            index=len(self.chain),
            data=data,
            previous_hash=prev_block.hash
        )
        self.chain.append(new_block)
        return new_block

    def is_valid(self):
        genesis = self.chain[0]
        if genesis.index != 0 or genesis.previous_hash != "0" * 64:
            return False
        if genesis.hash != genesis.calculate_hash():
            return False

        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i - 1]

            if current.hash != current.calculate_hash():
                return False

            if current.previous_hash != previous.hash:
                return False

        return True

print("Inisialisasi modul Block dan Blockchain berhasil!")

Inisialisasi modul Block dan Blockchain berhasil!


## Skenario 1: Inisialisasi Chain & Genesis Block
**Kriteria:** Chain baru hanya memiliki genesis block secara otomatis dan berstatus **Valid (`True`)**.

In [2]:
# Inisialisasi Blockchain Baru
bc = Blockchain()

print("Jumlah blok saat ini :", len(bc.chain))
print("Genesis Block Index  :", bc.chain[0].index)
print("Genesis Prev Hash    :", bc.chain[0].previous_hash)
print("Genesis Block Hash   :", bc.chain[0].hash)
print("Genesis Block Data   :", bc.chain[0].data)
print("Status Validasi Chain:", bc.is_valid())

Jumlah blok saat ini : 1
Genesis Block Index  : 0
Genesis Prev Hash    : 0000000000000000000000000000000000000000000000000000000000000000
Genesis Block Hash   : 925f311f6e1ba805d44d194b959114a331d2cceabdcc71b5e4c3e3d5c882f73b
Genesis Block Data   : {'keterangan': 'Genesis Block ITTS Blockchain'}
Status Validasi Chain: True


## Skenario 2: Menambahkan Minimal 3 Blok Transaksi
**Kriteria:** Menambahkan minimal 3 blok dengan payload transaksi berbeda, relasi hash valid, dan status `is_valid()` bernilai **`True`**.

In [3]:
t1 = {"pengirim": "Alice", "penerima": "Bob", "nominal": 50, "token": "IDR"}
t2 = {"pengirim": "Bob", "penerima": "Charlie", "nominal": 20, "token": "IDR"}
t3 = {"pengirim": "Charlie", "penerima": "Dave", "nominal": 15, "token": "IDR"}

b1 = bc.add_block(t1)
b2 = bc.add_block(t2)
b3 = bc.add_block(t3)

print("Blok 1 berhasil ditambah:", b1)
print("Blok 2 berhasil ditambah:", b2)
print("Blok 3 berhasil ditambah:", b3)
print("Total blok sekarang      :", len(bc.chain))
print("Status Validasi Chain    :", bc.is_valid())

Blok 1 berhasil ditambah: Block(Index=1, Hash=268e04b0bd5f8db5..., PrevHash=925f311f6e1ba805...)
Blok 2 berhasil ditambah: Block(Index=2, Hash=f767425b5a56c95c..., PrevHash=268e04b0bd5f8db5...)
Blok 3 berhasil ditambah: Block(Index=3, Hash=2a20d2f8252b947d..., PrevHash=f767425b5a56c95c...)
Total blok sekarang      : 4
Status Validasi Chain    : True


## Skenario 3: Tampering Data pada Blok Tengah (Block Index 2)
**Kriteria:** Mengubah isi data transaksi pada Blok 2 tanpa menghitung ulang chain. Program harus mendeteksi modifikasi dan menghasilkan **`False`**.

In [4]:
data_asli = bc.chain[2].data
print("Data asli Blok 2 :", data_asli)

# Modifikasi data transaksi secara sepihak
bc.chain[2].data = {"pengirim": "Bob", "penerima": "Attacker", "nominal": 9999, "token": "IDR"}
print("Data palsu Blok 2:", bc.chain[2].data)

status_tamper = bc.is_valid()
print("Status Validasi setelah Tampering Data:", status_tamper)

# Pulihkan data asli agar rantai kembali valid
bc.chain[2].data = data_asli
print("Status Validasi setelah data dipulihkan :", bc.is_valid())

Data asli Blok 2 : {'pengirim': 'Bob', 'penerima': 'Charlie', 'nominal': 20, 'token': 'IDR'}
Data palsu Blok 2: {'pengirim': 'Bob', 'penerima': 'Attacker', 'nominal': 9999, 'token': 'IDR'}
Status Validasi setelah Tampering Data: False
Status Validasi setelah data dipulihkan : True


## Skenario 4: Tampering `previous_hash` pada Blok Tengah
Menguji dua varian manipulasi relasi hash pada Blok 2:
- **Varian 4A:** `previous_hash` diubah TANPA update hash internal blok (Internal hash mismatch).
- **Varian 4B:** `previous_hash` diubah DENGAN update hash internal blok (Chain linkage broken).
**Expected Behavior:** Keduanya harus menghasilkan **`False`**.

In [5]:
original_prev_hash = bc.chain[2].previous_hash
original_hash = bc.chain[2].hash
fake_prev_hash = "deadbeef" * 8

# Varian 4A: previous_hash diubah tanpa update hash blok
print("--- [4A] previous_hash diubah TANPA update hash blok ---")
bc.chain[2].previous_hash = fake_prev_hash
print("Previous Hash palsu:", bc.chain[2].previous_hash)
print("Status Validasi (4A):", bc.is_valid())

# Pulihkan chain
bc.chain[2].previous_hash = original_prev_hash
bc.chain[2].hash = original_hash

# Varian 4B: previous_hash diubah DENGAN update hash blok
print("\n--- [4B] previous_hash diubah DENGAN update hash blok ---")
bc.chain[2].previous_hash = fake_prev_hash
bc.chain[2].hash = bc.chain[2].calculate_hash()
print("Previous Hash palsu:", bc.chain[2].previous_hash)
print("Hash dihitung ulang :", bc.chain[2].hash)
print("Status Validasi (4B):", bc.is_valid())

# Pulihkan chain kembali
bc.chain[2].previous_hash = original_prev_hash
bc.chain[2].hash = original_hash
print("Status Validasi setelah chain dipulihkan:", bc.is_valid())

--- [4A] previous_hash diubah TANPA update hash blok ---
Previous Hash palsu: deadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeef
Status Validasi (4A): False

--- [4B] previous_hash diubah DENGAN update hash blok ---
Previous Hash palsu: deadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeefdeadbeef
Hash dihitung ulang : 94066da82831a7c961a986f40e62adbaf22d0c2f37ab934d8b561d9585197fcb
Status Validasi (4B): False
Status Validasi setelah chain dipulihkan: True


## Skenario 5: Cetak Rincian Seluruh Blok dalam Blockchain
**Kriteria:** Menampilkan rincian seluruh blok (`index`, `timestamp`, `data`, `previous_hash`, dan `hash`) untuk membuktikan keterikatan rantai (*cryptographic linkage*).

In [6]:
print("=" * 75)
for block in bc.chain:
    print(f"Index         : {block.index}")
    print(f"Timestamp     : {block.timestamp}")
    print(f"Data          : {block.data}")
    print(f"Previous Hash : {block.previous_hash}")
    print(f"Current Hash  : {block.hash}")
    print("-" * 75)

Index         : 0
Timestamp     : 1793856085.7594466
Data          : {'keterangan': 'Genesis Block ITTS Blockchain'}
Previous Hash : 0000000000000000000000000000000000000000000000000000000000000000
Current Hash  : 925f311f6e1ba805d44d194b959114a331d2cceabdcc71b5e4c3e3d5c882f73b
---------------------------------------------------------------------------
Index         : 1
Timestamp     : 1793856085.7597742
Data          : {'pengirim': 'Alice', 'penerima': 'Bob', 'nominal': 50, 'token': 'IDR'}
Previous Hash : 925f311f6e1ba805d44d194b959114a331d2cceabdcc71b5e4c3e3d5c882f73b
Current Hash  : 268e04b0bd5f8db50b64822cb7cda98d0477e0352a6129ef0304f09c06e38c5e
---------------------------------------------------------------------------
Index         : 2
Timestamp     : 1793856085.7598011
Data          : {'pengirim': 'Bob', 'penerima': 'Charlie', 'nominal': 20, 'token': 'IDR'}
Previous Hash : 268e04b0bd5f8db50b64822cb7cda98d0477e0352a6129ef0304f09c06e38c5e
Current Hash  : f767425b5a56c95caefe6e6fe1

## Analisis Ilmiah: Mengapa Tampering Dapat Dideteksi?

1. **Efek Longsoran Salju (*Avalanche Effect*) SHA-256:**
   Fungsi hash SHA-256 bersifat deterministik dan satu arah (*preimage resistance*). Perubahan 1 karakter atau 1 bit pada data transaksi menghasilkan digest hash yang berubah total secara acak. Saat data Blok 2 diubah tanpa menghitung ulang hash (Skenario 3 & 4A), fungsi `is_valid()` mendeteksi ketidaksesuaian `current.hash != current.calculate_hash()`, sehingga validasi langsung ditolak (`False`).

2. **Keterikatan Kriptografis Rantai (*Cryptographic Linkage*):**
   Setiap blok menyimpan `previous_hash` yang merujuk pada hash blok pendahulunya. Jika penyerang memperbarui hash internal Blok 2 agar konsisten secara internal (Skenario 4B), nilai hash Blok 2 pasti berubah. Akibatnya, pada Blok 3 kondisi `current.previous_hash == previous.hash` menjadi tidak terpenuhi. Seluruh rantai sesudah blok yang dimanipulasi menjadi tidak valid (*broken chain*).